# SocialDilemmaJ

> JAX version of the symmetric 2-agent 2-action Social Dilemma Matrix Game.

In [ ]:
#| default_exp Environments/SocialDilemmaJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pyCRLD.Environments.BaseJ import ebaseJ
from fastcore.utils import *
from fastcore.test import *
import jax.numpy as jnp
import numpy as np

In [ ]:
#| export
class SocialDilemmaJ(ebaseJ):
    """
    Symmetric 2-agent 2-action Social Dilemma Matrix Game.
    """

    def __init__(self,
                 R: float,  # reward of mutual cooperation
                 T: float,  # temptation of unilateral defection
                 S: float,  # sucker's payoff of unilateral cooperation
                 P: float):  # punishment of mutual defection
        self.N = 2
        self.M = 2
        self.Z = 1

        self.Re = R
        self.Te = T
        self.Su = S
        self.Pu = P

        self.state = 0
        super().__init__()

In [ ]:
#| export
@patch
def TransitionTensor(self: SocialDilemmaJ):
    """Get the Transition Tensor."""
    return jnp.ones((self.Z, self.M, self.M, self.Z))

In [ ]:
#| export
@patch
def RewardTensor(self: SocialDilemmaJ):
    """Get the Reward Tensor R[i,s,a1,...,aN,s']."""
    R = jnp.zeros((2, self.Z, 2, 2, self.Z))
    R = R.at[0, 0, :, :, 0].set(jnp.array([[self.Re, self.Su],
                                             [self.Te, self.Pu]]))
    R = R.at[1, 0, :, :, 0].set(jnp.array([[self.Re, self.Te],
                                             [self.Su, self.Pu]]))
    return R

In [ ]:
#| export
@patch
def actions(self: SocialDilemmaJ):
    """The action sets"""
    return [['c', 'd'] for _ in range(self.N)]

@patch
def states(self: SocialDilemmaJ):
    """The states set"""
    return ['.']

@patch
def id(self: SocialDilemmaJ):
    """
    Returns id string of environment
    """
    # Default
    id = f"{self.__class__.__name__}_"+\
        f"{self.Te}_{self.Re}_{self.Pu}_{self.Su}"
    return id

## Tests

In [ ]:
# Prisoner's Dilemma: T > R > P > S
env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)
assert env.T.shape == (1, 2, 2, 1)
assert env.R.shape == (2, 1, 2, 2, 1)
assert jnp.allclose(env.T.sum(-1), 1.0)
assert env.R[0, 0, 0, 0, 0] == 3.0  # mutual coop reward
assert env.R[0, 0, 1, 0, 0] == 5.0  # temptation

In [ ]:
# Check id
assert 'SocialDilemmaJ' in env.id()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()